In [1]:
import tensorflow as tf
import numpy as np
import gym
from ou_noise import OUNoise
import random


LAYER_1 = 400
LAYER_2 = 300
LAYER_3 = 300
keep_rate = 0.8
LAMBDA = 0.00001 # regularization term
GAMMA = 0.99
class DDPG(object):


    def __init__(self, sess, state_dim, action_dim, max_action, min_action, actor_learning_rate, critic_learning_rate, tau, RANDOM_SEED, device = '/cpu:0'):

        self.sess = sess
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        self.s_dim = state_dim
        self.a_dim = action_dim
        self.actor_learning_rate = actor_learning_rate
        self.critic_learning_rate = critic_learning_rate
        self.tau = tau
        self.device = device
        self.max_action = max_action
        self.min_action = min_action
        # Placeholders
        self.inputs = tf.placeholder(tf.float32, shape=[None, self.s_dim], name='state')
        self.action = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='actions')
        scope = 'net'    
        self.v, self.a, self.scaled_a, self.saver = self._build_net(scope)
        self.a_params = tf.trainable_variables(scope=scope + '/actor')
        self.c_params = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')
        scope = 'target'    
        self.v_target, self.a_target, self.scaled_a_target, self.saver_target = self._build_net(scope)
        self.a_params_target = tf.trainable_variables(scope=scope + '/actor')
        self.c_params_target = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')

        
        with tf.variable_scope('learning_rate'): 
            # global step
            self.global_step = tf.Variable(0, trainable=False)
            self.actor_decay_learning_rate = tf.train.exponential_decay(self.actor_learning_rate, self.global_step, 100000, 0.96, staircase=True)
            self.critic_decay_learning_rate = tf.train.exponential_decay(self.critic_learning_rate, self.global_step, 100000, 0.96, staircase=True)
        
        with tf.device(self.device):
            # Op for periodically updating target network with online network
            # weights with regularization
            self.generate_param_updater()
           
            self.predicted_q_value = tf.placeholder(tf.float32, [None, 1])
            # Define loss and optimization Op
            self.squared = tf.square(tf.subtract(self.predicted_q_value,self.v))
            self.l2_loss = tf.losses.get_regularization_loss(scope="net/critic")
            self.loss = tf.reduce_mean(self.squared) + self.l2_loss 
            self.critic_optimize = tf.train.AdamOptimizer(self.critic_decay_learning_rate).minimize(self.loss, global_step=self.global_step) 
            self.action_grads = tf.gradients(self.v, self.action)[0]
            self.actor_gradients = tf.gradients(self.a, self.a_params, -self.action_grads)
            self.actor_optimize = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self.actor_gradients, self.a_params), global_step=self.global_step)
            
            # inverting gradients
            self.inverting_gradients_placeholder = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='inverting_gradients')
            self._dq_da = tf.gradients(self.v, self.action)[0] # q, a 
            self._grad = tf.gradients(self.a, self.a_params, -self.inverting_gradients_placeholder)
            self._train_actor = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self._grad, self.a_params),global_step=self.global_step)
            
            


    def _build_net(self,scope):
       
        with tf.device(self.device):        
            with tf.variable_scope(scope + '/critic'):
                
                
                '''
                net = tf.layers.dense(self.inputs, LAYER_1, tf.nn.relu, name='critic_L1')
                initializer = tf.variance_scaling_initializer()
                s_union_weights = tf.Variable(initializer.__call__([LAYER_1, LAYER_2]), name='critic_L2_Ws')
                a_union_weights = tf.Variable(initializer.__call__([self.a_dim, LAYER_2]), name='critic_L2_Wa')
                union_biases = tf.Variable(tf.zeros([LAYER_2]), name='critic_L2_b')
                net = tf.nn.relu(tf.matmul(net, s_union_weights) + tf.matmul(self.action, a_union_weights) + union_biases,name='critic_L2')
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                v = tf.layers.dense(net, self.a_dim, kernel_initializer=w_init, name='critic_output')
                '''
                regularizer = tf.contrib.layers.l2_regularizer(scale=LAMBDA)
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1, weights_regularizer=regularizer, activation_fn=tf.nn.leaky_relu)
                l2_a = tf.contrib.layers.fully_connected(self.action, LAYER_2, weights_regularizer=regularizer, activation_fn=None)
                l2_s = tf.contrib.layers.fully_connected(l1, LAYER_2, weights_regularizer=regularizer,activation_fn=None)
                l2 = tf.nn.leaky_relu(l2_s + l2_a)
                v = tf.contrib.layers.fully_connected(l2, 1, weights_regularizer=regularizer, activation_fn=None)
				
            with tf.variable_scope(scope + '/actor'):
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1,  activation_fn=tf.nn.leaky_relu) # tf.nn.leaky_relu tf.nn.relu
                l2 = tf.contrib.layers.fully_connected(l1, LAYER_2,  activation_fn=tf.nn.leaky_relu)
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=tf.nn.tanh) # (para el ddpg)
                #a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=None) # (para el inverted)
                scaled_a = tf.multiply(a,self.max_action) #(para el ddpg)
                #scaled_a = a # (para el inverted)
                
                       
        saver = tf.train.Saver()
        return v, a, scaled_a, saver

    def train(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # print(L2_LOSS)
        a_outs = self.predict_action(s_batch)
        self.actor_train(s_batch, a_outs)
        
        self.update_target_network()

        return

    def test_gradient(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # train critic
        #ac_tor_grads = self._critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        #print('a grads',ac_tor_grads)
        actions = self.predict_action(s_batch)
        
        upper = self.max_action
        lower = self.min_action

        # get dq/da array, action array
        #print(upper, '***************')
        dq_das = self.sess.run([self._dq_da], feed_dict={self.inputs: s_batch, self.action:actions})[0]
        # inverting gradients, if dq_da >= 0, apply upper method, else lower method
        inverting_gradients = []
        #'''
        # print('1 dq_das, actions',dq_das, actions)
        '''
        # print('dq_das, actions',dq_das, actions)
        for dq_da, action in zip(dq_das, actions):
            # print('dq_da, action',dq_da, action)
            if dq_da >= 0.0:
                inverting_gradients.append(dq_da * (self.max_action - action) / (self.max_action - self.min_action))
            else:
                inverting_gradients.append(dq_da * (action - self.min_action) / (self.max_action - self.min_action))
        inverting_gradients = np.array(inverting_gradients).reshape(-1, 1)
		'''

        for i in range(MINIBATCH_SIZE):
            #print('2', i,dq_das[i])
            for j in range(self.a_dim):
                if dq_das[i][j] >= 0.0:
                    dq_das[i][j] = dq_das[i][j] * (self.max_action - actions[i][j]) / (self.max_action - self.min_action)
                else:
                    dq_das[i][j] = dq_das[i][j] * (actions[i][j] - self.min_action) / (self.max_action - self.min_action)
        
        # print(dq_das,inverting_gradients)
        # exit()
        inverting_gradients = dq_das 
        
        # print('2 dq_das, actions',dq_das, actions)
        
         
        #print('1','inverting_gradients',inverting_gradients)
        
        # print('2','inverting_gradients',inverting_gradients,dq_das, actions)
        # time.sleep(1)
        # update actor
        self.sess.run(self._train_actor, feed_dict={self.inputs: s_batch, self.inverting_gradients_placeholder: inverting_gradients})
        self.update_target_network()
        return

    
    def _critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.action_grads], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def update_target_network(self):
        self.sess.run([self.a_updater,self.c_updater])

    def generate_param_updater(self):
        self.a_updater = [self.a_params_target[i].assign(tf.multiply(self.a_params[i], self.tau) + tf.multiply(self.a_params_target[i], 1. - self.tau))
                for i in range(len(self.a_params))]
        self.c_updater = [self.c_params_target[i].assign(tf.multiply(self.c_params[i], self.tau) + tf.multiply(self.c_params_target[i], 1. - self.tau))
                for i in range(len(self.c_params))]

    def critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.loss,self.critic_optimize], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def actor_train(self,inputs, action):
        return self.sess.run(self.actor_optimize, feed_dict={
            self.inputs: inputs,
            self.action: action
        })

    def save(self):
        self.saver.save(self.sess,"./model/model.ckpt")
        self.saver_target.save(self.sess,"./model/model_target.ckpt")
        print("Model saved in file: actor_model")

    
    def load(self):
        self.saver.restore(self.sess,"./model/model.ckpt")
        self.saver_target.restore(self.sess,"./model/model_target.ckpt")
        


    def critic_predict_target(self, state, action):
        return self.sess.run(self.v_target, feed_dict={
            self.inputs: state,
            self.action: action
        })
        
    def predict_action_target(self, state):
        return self.sess.run(self.scaled_a_target, feed_dict={
            self.inputs: state
        })

    def predict_action(self, state):
        return self.sess.run(self.scaled_a, feed_dict={
            self.inputs: state
        })



def normalizing_state(state):
    #state = [V,P] being V_min = 0, V_max = 210, P_min = 0, P_max = 25000 we use ((xi-x_min)/(x_max-x_min))-1
    V_min = 0
    V_max = 210
    P_min = 0
    P_max = 54000
    DeltaP_min = -15000
    DeltaP_max = 15000

    st = [(2*(state[0]-V_min)/(V_max-V_min))-1, (2*(state[1]-P_min)/(P_max-P_min))-1,(2*(state[1]-DeltaP_min)/(DeltaP_max-DeltaP_min))-1]

    return st


if __name__ == '__main__':
    from replay_buffer import ReplayBuffer
    import gym
    import gym_mppt
    ACTOR_LEARNING_RATE = 0.0001
    CRITIC_LEARNING_RATE =  0.001
    # Soft target update param
    TAU = 0.001
    DEVICE = '/cpu:0'
    # ENV_NAME = 'MountainCarContinuous-v0'
    ENV_NAME = 'mppt_shaded-v0'#'Pendulum-v0'
    # import gym_foo
    # ENV_NAME = 'nessie_end_to_end-v0'
    max_action = 10.
    min_action = -10.
    epochs = 15000
#     epochs = 20000
    epsilon = 1.0
    min_epsilon = 0.1
    EXPLORE = 15000
    BUFFER_SIZE = 50000
    RANDOM_SEED = 51234
    MINIBATCH_SIZE = 64# 32 # 5
    with tf.Session() as sess:
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        env = gym.make(ENV_NAME)
        state_dim = np.size(env.reset()) #env.observation_space.shape[0]
        action_dim = 1 #env.action_space.shape[0]
        ddpg = DDPG(sess, state_dim, action_dim, max_action, min_action, ACTOR_LEARNING_RATE, CRITIC_LEARNING_RATE, TAU, RANDOM_SEED,device=DEVICE)
        sess.run(tf.global_variables_initializer())
        #ddpg.load()
        replay_buffer = ReplayBuffer(BUFFER_SIZE, RANDOM_SEED)
        ruido = OUNoise(action_dim, mu = 0.0)
        Reward_episodios = []
        llegadas =0
        for i in range(epochs):
            state = env.reset()
            normalized_state = normalizing_state(state)
            #print('EL ESTADO RESETEADO ES', state, state.shape)
            #print('epoch = ', i)
            done = False
            epsilon -= (epsilon/EXPLORE)
            epsilon = np.maximum(min_epsilon,epsilon)
            episode_r = 0.
            step = 0 
            max_steps = 45 #es solo a efecto de contador
            r_episodio_actual = []
            #llegadas =0
            while (not done):
                #print('step =', step)
                #wait = input("PRESS ENTER TO CONTINUE.")
                action = ddpg.predict_action(np.reshape(normalized_state,(1,state_dim)))
                action1 = action
                #print('LA ACCION sin clipear ES', action1, action1.shape) 
                action = np.clip(action1,min_action,max_action)
                #action = action + max(epsilon,0)*ruido.noise()
                #e-greedy action selection
                if random.random() < epsilon:
                    action = random.uniform(min_action,max_action)
                    #print('Explorando')

                action = np.clip(action,min_action,max_action)
                #print('ruido =', max(epsilon,0)*ruido.noise(),'epsilon =',epsilon)
                #print('LA ACCION clipeada ES', action, action.shape)
                
                next_state, reward, done, info = env.step(action)
                normalized_next_state = normalizing_state(next_state)
                #print('EL NEXT_ESTADO ES', next_state, next_state.shape) 
                #reward = np.clip(reward,-1.,1.)
                #print('instaneous r = ',reward)
                #print('Steps = ', info['Steps'],'V0 =',info['v0'])
                #print('normalized_state =',normalized_state)
                #wait = input("PRESS ENTER TO CONTINUE.")
                replay_buffer.add(np.reshape(normalized_state, (state_dim,)), np.reshape(action, (action_dim,)), reward,
                                      done, np.reshape(normalized_next_state, (state_dim,)))
                state = next_state
                episode_r = episode_r + reward
                normalized_state = normalizing_state(state)

                r_episodio_actual.append(reward)
                if replay_buffer.size() > MINIBATCH_SIZE:
                    s_batch, a_batch, r_batch, t_batch, s2_batch = replay_buffer.sample_batch(MINIBATCH_SIZE)
                    # train ddpg normally:
                    ddpg.train(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
                    #train with inverted gradients
                    #ddpg.test_gradient(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
#                 print(i, step, 'last r', round(reward,3), 'episode reward',round(episode_r,3), 'epsilon', round(epsilon,3))
#                 print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state,'last r =', round(reward[0][0],3), 'episode reward =',round(episode_r[0][0],3), 'epsilon =', round(epsilon,3))
                
                step += 1
            
            
            #if info['Steps'] > max_steps:
             #   llegadas +=1
#             print ('--------------------------------------------')
#             print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state, 'accion =',action,'last r =', reward, 'episode reward =',episode_r, 'epsilon =', round(epsilon,3))
#             print ('--------------------------------------------')

                

            Reward_episodios.append(r_episodio_actual)
            np.save('Reward_episodios_DDPG01.npy',Reward_episodios)
            

            print('epoch          = ', i)      
            print('Episode Reward = ', episode_r/10)
            print('Last Reward    = ', reward)
            print('State          = ', state)
            print()

            if i % 1000 == 0:      # Save model every 1000 iterations
                ddpg.save()
            
        ddpg.save()

C:\Users\archi\anaconda3\envs\BTP\lib\site-packages\gym\logger.py:30: UserWarning: WARN: Box bound precision lowered by casting to float32
  warnings.warn(colorize('%s: %s'%('WARN', msg % args), 'yellow'))


epoch          =  0
Episode Reward =  7.054388000000005
Last Reward    =  9e-05
State          =  [   7.4044185 2780.0706    2780.0706   ]

Model saved in file: actor_model
epoch          =  1
Episode Reward =  1.2742890000000004
Last Reward    =  0.5759
State          =  [   93.281364 21959.31     -1390.8065  ]

epoch          =  2
Episode Reward =  10.619585999999996
Last Reward    =  0.04793
State          =  [  189.85744 15215.961   -1858.0735 ]

epoch          =  3
Episode Reward =  9.13436195708561
Last Reward    =  0.8084499835968018
State          =  [  168.83066 21251.404    1583.8047 ]

epoch          =  4
Episode Reward =  3.8656890000000006
Last Reward    =  0.06528
State          =  [  15.310417 5742.3335   -675.31744 ]

epoch          =  5
Episode Reward =  0.9706039999999992
Last Reward    =  0.06151
State          =  [   41.48672 15562.327   -1992.7031 ]

epoch          =  6
Episode Reward =  5.549263
Last Reward    =  0.04241
State          =  [  151.97672 12598.614   

c:\users\archi\python code\btp\luis_upgrade\mppt-gym_v4\gym_mppt\envs\mppt_env_shaded0.py:37: RuntimeWarning: overflow encountered in exp
  I = self.Np * Iph - self.Np * Irs * (np.exp(self.q / (self.k * Tcell * self.A) * vx * (1/SH) / self.Ns) - 1)


epoch          =  42
Episode Reward =  3.5295830028817945
Last Reward    =  0.0
State          =  [187.46935   0.        0.     ]

epoch          =  43
Episode Reward =  0.5804740000000002
Last Reward    =  0.05434
State          =  [   69.31907 26000.078   -2780.0818 ]

epoch          =  44
Episode Reward =  1.6971940001729753
Last Reward    =  0.15722
State          =  [   57.1956 13701.461  -1653.4545]

epoch          =  45
Episode Reward =  6.720862999999999
Last Reward    =  0.61245
State          =  [  117.958374 16551.85        87.53967 ]

epoch          =  46
Episode Reward =  1.1470219985850238
Last Reward    =  0.0077
State          =  [   24.888636  9336.581    -3601.2317  ]

epoch          =  47
Episode Reward =  4.090702999999999
Last Reward    =  0.1971
State          =  [   82.14341 13555.617    -633.6759 ]

epoch          =  48
Episode Reward =  12.423947998608588
Last Reward    =  0.8380600214004517
State          =  [  130.37418 20493.39     -330.80664]

epoch        

epoch          =  99
Episode Reward =  2.4372560000000014
Last Reward    =  0.0017
State          =  [ 142.16537 3080.0005  1280.7607 ]

epoch          =  100
Episode Reward =  5.194878011919432
Last Reward    =  0.00349
State          =  [  12.168383 4553.2744   2235.6536  ]

epoch          =  101
Episode Reward =  5.247664999995548
Last Reward    =  0.09054
State          =  [   32.827637 12293.691     2683.8796  ]

epoch          =  102
Episode Reward =  1.772512
Last Reward    =  0.02631
State          =  [   28.389347 10653.568     3237.6174  ]

epoch          =  103
Episode Reward =  6.145541007509394
Last Reward    =  0.0
State          =  [209.20935   0.        0.     ]

epoch          =  104
Episode Reward =  5.483287011672776
Last Reward    =  0.30498000979423523
State          =  [  103.00113 11164.755    -336.18262]

epoch          =  105
Episode Reward =  5.409962996536139
Last Reward    =  0.0117
State          =  [   32.569088 12225.392     3089.1782  ]

epoch          =

...

epoch          =  596
Episode Reward =  10.816499009572691
Last Reward    =  0.07416000217199326
State          =  [   31.443146 11790.483     2630.9414  ]

epoch          =  597
Episode Reward =  7.568304007671969
Last Reward    =  0.6171799898147583
State          =  [   33.26369 11734.89      383.46973]

epoch          =  598
Episode Reward =  4.424682013003251
Last Reward    =  0.6740300059318542
State          =  [  136.24612 21005.129    -412.64648]

epoch          =  599
Episode Reward =  6.539592014854685
Last Reward    =  0.0
State          =  [212.   0.   0.]

epoch          =  600
Episode Reward =  19.176954999729308
Last Reward    =  0.6367400288581848
State          =  [   84.386284 30933.352    -2100.293   ]

epoch          =  601
Episode Reward =  16.554928992044616
Last Reward    =  0.48475998640060425
State          =  [   67.29824  10020.358      -15.628906]

epoch          =  602
Episode Reward =  8.538398969529155
Last Reward    =  0.6807000041007996
State          

epoch          =  651
Episode Reward =  10.043667975575199
Last Reward    =  0.011130000464618206
State          =  [   20.637125  7746.8296   -2110.401   ]

epoch          =  652
Episode Reward =  13.815798012879805
Last Reward    =  0.4040200114250183
State          =  [   47.59106 17160.557     207.82227]

epoch          =  653
Episode Reward =  7.495472016833185
Last Reward    =  0.006599999964237213
State          =  [  21.709988 5215.8247   -181.49023 ]

epoch          =  654
Episode Reward =  16.031733035698554
Last Reward    =  0.0
State          =  [212.   0.   0.]

epoch          =  655
Episode Reward =  0.6904459972966805
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  656
Episode Reward =  4.101115993588881
Last Reward    =  0.12488000094890594
State          =  [   73.12673 12093.988     700.51855]

epoch          =  657
Episode Reward =  7.6445529703554715
Last Reward    =  0.9213100075721741
State          =  [  107.4265 37866.02     642.8086]

epoc

epoch          =  706
Episode Reward =  5.573787008406688
Last Reward    =  0.09689000248908997
State          =  [   33.48615 12503.972    -567.4346 ]

epoch          =  707
Episode Reward =  6.246481983908656
Last Reward    =  0.0
State          =  [191.25142   0.        0.     ]

epoch          =  708
Episode Reward =  7.572658996971777
Last Reward    =  0.45120999217033386
State          =  [   96.95259 35925.914    3156.9883 ]

epoch          =  709
Episode Reward =  1.7095580017641865
Last Reward    =  0.030419999733567238
State          =  [   53.791294 12915.498     -949.7754  ]

epoch          =  710
Episode Reward =  12.118859982215925
Last Reward    =  0.9999600052833557
State          =  [   89.10513 19208.748    1334.873  ]

epoch          =  711
Episode Reward =  14.235845991065053
Last Reward    =  0.5185199975967407
State          =  [   86.785095 21067.45       411.78125 ]

epoch          =  712
Episode Reward =  18.68334007123763
Last Reward    =  4.999999873689376e-0

epoch          =  761
Episode Reward =  0.7698230001828936
Last Reward    =  0.04335999861359596
State          =  [  40.208443 4493.3027   -246.14844 ]

epoch          =  762
Episode Reward =  19.9506310261786
Last Reward    =  0.8615300059318542
State          =  [   73.838425 11643.177     -432.5088  ]

epoch          =  763
Episode Reward =  13.556358984217644
Last Reward    =  0.016990000382065773
State          =  [  14.940912 5472.742    -187.36133 ]

epoch          =  764
Episode Reward =  5.0529150024259195
Last Reward    =  0.7684599757194519
State          =  [  104.13049 24765.803    -568.7988 ]

epoch          =  765
Episode Reward =  11.459429986332308
Last Reward    =  0.9026399850845337
State          =  [  110.48541 39129.816    -754.5664 ]

epoch          =  766
Episode Reward =  7.804020015662678
Last Reward    =  0.3849399983882904
State          =  [  114.14906 42418.13     -971.9531 ]

epoch          =  767
Episode Reward =  11.593557954828768
Last Reward    =  0.

KeyboardInterrupt: 

In [ ]:
# 7500  -> 100 minutes (1 hour 40 mins)
# 15000 -> 200 minutes (3 hours 20 mins) # X around 9 hours